# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [2]:
import sys
!{sys.executable} -m pip install google-generativeai


In [3]:
!pip install python-dotenv


In [4]:
from dotenv import load_dotenv
import os

load_dotenv(".env")
api_key = os.getenv('Gemini_API_KEY')

In [6]:
from google import genai

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
model="gemini-3-flash-preview",
contents="Explain how AI works in a few words"
)

print(response.text)

AI analyzes vast amounts of data to find **patterns** and make **predictions**.


## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [7]:
from sklearn.datasets import fetch_20newsgroups

datos_grupos = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
docs_originales = datos_grupos.data

import pandas as pd

df = pd.DataFrame(docs_originales, columns=['text'])
df

,text
0,\n\nI am sure some bashers of Pens fans are pr...
1,My brother is in the market for a high-perform...
2,\n\n\n\n\tFinally you said what you dream abou...
3,\nThink!\n\nIt's the SCSI card doing the DMA t...
4,1) I have an old Jasmine drive which I cann...
...,...
18841,DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...
18842,\nNot in isolated ground recepticles (usually ...
18843,I just installed a DX2-66 CPU in a clone mothe...
18844,\nWouldn't this require a hyper-sphere. In 3-...


### 2.2 Transformo a embeddings

In [9]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s


df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,text,text_norm
0,\n\nI am sure some bashers of Pens fans are pr...,I am sure some bashers of Pens fans are pretty...
1,My brother is in the market for a high-perform...,My brother is in the market for a high-perform...
2,\n\n\n\n\tFinally you said what you dream abou...,Finally you said what you dream about. Mediter...
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,Think! It's the SCSI card doing the DMA transf...
4,1) I have an old Jasmine drive which I cann...,1) I have an old Jasmine drive which I cannot ...


In [11]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()

        if len(chunk) > 0:
            chunks.append(chunk)

        if end == n:
            break

        start = max(0, end - overlap)

    return chunks


records = []

for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)

    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)

chunks_df.head(), len(chunks_df)


(   doc_id  chunk_id                                               text
 0       0         0  I am sure some bashers of Pens fans are pretty...
 1       1         0  My brother is in the market for a high-perform...
 2       2         0  Finally you said what you dream about. Mediter...
 3       2         1  urds and Turks once upon a time! Ohhhh so swed...
 4       3         0  Think! It's the SCSI card doing the DMA transf...,
 38871)

In [12]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2" # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

C:\Users\LabP4E010\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LabP4E010\.cache\huggingface\hub\models--intfloat--e5-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [38]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
passages[:1000],
batch_size=16,
show_progress_bar=True,
convert_to_numpy=True,
normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

### 2.3 Creo una query y hago la búsqueda

In [40]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "baseball balls"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

Obtengo los 5 documentos más similares a mi query

In [41]:
import numpy as np

scores = np.dot(embeddings, query_vec.T).flatten()

top_k_indices = np.argsort(scores)[::-1][:5]

print(f"Resultados para la búsqueda: '{query_text}'\n")
for i, idx in enumerate(top_k_indices):
    score = scores[idx]
    text = chunks_df.iloc[idx]["text"]
    print(f"Top {i+1} (Score: {score:.4f})")
    print(f"{text[:300]}...\n")

Resultados para la búsqueda: 'baseball balls'

Top 1 (Score: 0.8165)
I don't Well, no problem! But I get pretty annoyed when they swing at non-strikes and make outs. Especially ball four on the 3-2 counts... Dave...

Top 2 (Score: 0.8062)
bosox-request@world.std.com to mail to the list: bosox@world.std.com...

Top 3 (Score: 0.8028)
' Law. Don't believe in catchers' era. But I am interested in pitchers' eras with different catchers. Any info on that? In other words, we know more than they do, so the only logic behind a different decision than we would make must be financial. I presume we feel this way about other franchises tha...

Top 4 (Score: 0.7987)
nd Indians 3 Los Angeles Dodgers 2 Boston Red Sox 4 (13) Atlanta Braves 1 California Angels PPD San Francisco Giants 6 Milwaukee Brewers RAIN Chicago Cubs IDLE Baltimore Orioles IDLE Cincinnati Reds IDLE Chicago White Sox IDLE Florida Marlins IDLE Minnesota Twins IDLE Philadelphia PhilliesIDLE Texas...

Top 5 (Score: 0.7955)
Won 3 03-00 

In [44]:

import numpy as np

scores = np.dot(embeddings, query_vec.T).flatten()
top_k_indices = np.argsort(scores)[::-1][:5]

top_docs_text = ""

for i, idx in enumerate(top_k_indices):
    text = chunks_df.iloc[idx]["text"]
    top_docs_text += f"\nDocumento {i+1}:\n{text}\n"

prompt = f"""
Eres un asistente de inteligencia artificial especializado en recuperación de información (RAG).

Consulta del usuario:
"{query_text}"

A continuación se presentan los 5 fragmentos de documentos más relevantes recuperados mediante embeddings.
Los textos pueden contener información parcial o ruido.

Tareas:
1. Explica brevemente por qué estos documentos son relevantes para la consulta.
2. Elabora un resumen claro que responda a la consulta utilizando ÚNICAMENTE la información presente en los documentos.
3. No inventes información que no esté en los textos.

Documentos recuperados:
{top_docs_text}

Responde en español, con un tono claro y explicativo, adecuado para un trabajo universitario.
"""

from google import genai

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt
)

print(response.text)


Estimado estudiante, a continuación presento el análisis de la información recuperada en relación con su consulta sobre **"baseball balls"**.

### 1. Relevancia de los documentos
Los documentos seleccionados son relevantes para la consulta por las siguientes razones:
*   **Terminología técnica:** El Documento 1 utiliza el término "ball" en el contexto reglamentario del juego (específicamente "ball four" y la cuenta de "3-2"), diferenciándolo de los "strikes".
*   **Contexto deportivo:** Los documentos 3, 4 y 5 sitúan la temática dentro del ámbito profesional de las Grandes Ligas de Béisbol (MLB), mencionando equipos, estadísticas de lanzadores (*pitchers*) y receptores (*catchers*), y resultados de partidos.
*   **Identificación de entidades:** Se identifican franquicias específicas (como los Boston Red Sox o los Atlanta Braves) y la estructura organizativa del deporte (National League y American League).

---

### 2. Resumen de la información recuperada
Basándose exclusivamente en los